In [1]:
# ============================================================
# DEEP LEARNING-BASED SENTIMENT ANALYSIS FOR SOCIAL IMPACT
# BERT-based 4-class sentiment classification
# ============================================================

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
accuracy_score,
classification_report,
confusion_matrix,
precision_recall_fscore_support
)

from datasets import Dataset

from transformers import (
BertTokenizer,
BertForSequenceClassification,
Trainer,
TrainingArguments
)

c:\Users\sujay\anaconda3\Lib\site-packages\transformers\utils\generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [ ]:
# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATA_FILE = "community_feedback.csv"

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

MODEL_NAME = "bert-base-uncased"

NUM_LABELS = 4

LABEL_NAMES = [
"positive",
"neutral",
"negative",
"distress"
]





In [ ]:
# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

print("Loading dataset...")

df = pd.read_csv(DATA_FILE)

print("\nDataset shape:")
print(df.shape)

print("\nFirst five rows:")
print(df.head())

print("\nColumn names:")
print(df.columns)

In [ ]:
# ------------------------------------------------------------
# 3. BASIC DATA CLEANING
# ------------------------------------------------------------

# Keep only required columns
df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

# Remove missing values
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

# Convert text to string
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

# Convert labels to lowercase strings
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.lower().str.strip()

# Remove empty text
df = df[df[TEXT_COLUMN].str.strip() != ""]

# Keep only valid labels
df = df[df[LABEL_COLUMN].isin(LABEL_NAMES)]

print("\nDataset after cleaning:")
print(df.shape)

print("\nClass distribution:")
print(df[LABEL_COLUMN].value_counts())

In [ ]:
# ------------------------------------------------------------
# 4. CONVERT TEXT LABELS TO NUMERICAL LABELS
# ------------------------------------------------------------

label2id = {
"positive": 0,
"neutral": 1,
"negative": 2,
"distress": 3
}

id2label = {
0: "positive",
1: "neutral",
2: "negative",
3: "distress"
}

df["label_id"] = df[LABEL_COLUMN].map(label2id)

print("\nEncoded labels:")
print(df[[TEXT_COLUMN, LABEL_COLUMN, "label_id"]].head())

# ------------------------------------------------------------
# 5. TRAIN / VALIDATION / TEST SPLIT
# ------------------------------------------------------------

# First split:
# 80% training
# 20% temporary data

train_df, temp_df = train_test_split(
df,
test_size=0.20,
random_state=42,
stratify=df["label_id"]
)

# Split remaining 20% into:
# 10% validation
# 10% test

validation_df, test_df = train_test_split(
temp_df,
test_size=0.50,
random_state=42,
stratify=temp_df["label_id"]
)

print("\nTraining samples:", len(train_df))
print("Validation samples:", len(validation_df))
print("Testing samples:", len(test_df))

In [ ]:
# ------------------------------------------------------------
# 6. CONVERT TO HUGGING FACE DATASETS
# ------------------------------------------------------------

train_dataset = Dataset.from_pandas(
train_df[[TEXT_COLUMN, "label_id"]],
preserve_index=False
)

validation_dataset = Dataset.from_pandas(
validation_df[[TEXT_COLUMN, "label_id"]],
preserve_index=False
)

test_dataset = Dataset.from_pandas(
test_df[[TEXT_COLUMN, "label_id"]],
preserve_index=False
)

# Rename label column because Hugging Face Trainer expects
# the target column to be called "labels"

train_dataset = train_dataset.rename_column(
"label_id",
"labels"
)

validation_dataset = validation_dataset.rename_column(
"label_id",
"labels"
)

test_dataset = test_dataset.rename_column(
"label_id",
"labels"
)

# ------------------------------------------------------------
# 7. LOAD BERT TOKENIZER
# ------------------------------------------------------------

print("\nLoading BERT tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
MODEL_NAME
)

In [ ]:
# ------------------------------------------------------------
# 8. TOKENIZATION FUNCTION
# ------------------------------------------------------------

def tokenize_function(examples):

return tokenizer(
examples[TEXT_COLUMN],
padding="max_length",
truncation=True,
max_length=128
)


print("\nTokenizing dataset...")

train_dataset = train_dataset.map(
tokenize_function,
batched=True
)

validation_dataset = validation_dataset.map(
tokenize_function,
batched=True
)

test_dataset = test_dataset.map(
tokenize_function,
batched=True
)

# ------------------------------------------------------------
# 9. LOAD PRE-TRAINED BERT MODEL
# ------------------------------------------------------------

print("\nLoading BERT model...")

model = BertForSequenceClassification.from_pretrained(
MODEL_NAME,
num_labels=NUM_LABELS,
id2label=id2label,
label2id=label2id
)


In [ ]:

# ------------------------------------------------------------
# 10. EVALUATION METRICS
# ------------------------------------------------------------

def compute_metrics(eval_prediction):

logits, labels = eval_prediction

predictions = np.argmax(
logits,
axis=1
)

accuracy = accuracy_score(
labels,
predictions
)

precision, recall, f1, _ = precision_recall_fscore_support(
labels,
predictions,
average="weighted",
zero_division=0
)

return {
"accuracy": accuracy,
"precision": precision,
"recall": recall,
"f1": f1
}

# ------------------------------------------------------------
# 11. TRAINING CONFIGURATION
# ------------------------------------------------------------

training_args = TrainingArguments(

output_dir="./bert_sentiment_model",

eval_strategy="epoch",

save_strategy="epoch",

learning_rate=2e-5,

per_device_train_batch_size=16,

per_device_eval_batch_size=16,

num_train_epochs=3,

weight_decay=0.01,

logging_dir="./logs",

logging_steps=50,

load_best_model_at_end=True,

metric_for_best_model="accuracy",

greater_is_better=True,

report_to="none"
)


In [ ]:
# ------------------------------------------------------------
# 12. CREATE TRAINER
# ------------------------------------------------------------

trainer = Trainer(

model=model,

args=training_args,

train_dataset=train_dataset,

eval_dataset=validation_dataset,

tokenizer=tokenizer,

compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 13. TRAIN BERT MODEL
# ------------------------------------------------------------

print("\nStarting BERT training...")

trainer.train()

print("\nTraining completed.")

# ------------------------------------------------------------
# 14. EVALUATE ON TEST DATA
# ------------------------------------------------------------

print("\nEvaluating model on test data...")

test_results = trainer.evaluate(
test_dataset
)

print("\nTest results:")
print(test_results)

test_accuracy = test_results["eval_accuracy"]

print(
"\nTest Accuracy: "
f"{test_accuracy * 100:.2f}%"
)

In [ ]:
# ------------------------------------------------------------
# 15. GENERATE PREDICTIONS
# ------------------------------------------------------------

print("\nGenerating predictions...")

prediction_output = trainer.predict(
test_dataset
)

logits = prediction_output.predictions

predicted_labels = np.argmax(
logits,
axis=1
)

actual_labels = prediction_output.label_ids

# ------------------------------------------------------------
# 16. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report:")

print(
classification_report(
actual_labels,
predicted_labels,
target_names=LABEL_NAMES,
zero_division=0
)
)

# ------------------------------------------------------------
# 17. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
actual_labels,
predicted_labels
)

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(8, 6))

sns.heatmap(
cm,
annot=True,
fmt="d",
cmap="Blues",
xticklabels=LABEL_NAMES,
yticklabels=LABEL_NAMES
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("BERT Sentiment Classification - Confusion Matrix")

plt.tight_layout()

plt.show()

# ------------------------------------------------------------
# 18. ADD PREDICTIONS TO TEST DATA
# ------------------------------------------------------------

test_results_df = test_df.copy()

test_results_df["predicted_label"] = [
id2label[x]
for x in predicted_labels
]

print("\nPrediction examples:")

print(
test_results_df[
[
TEXT_COLUMN,
LABEL_COLUMN,
"predicted_label"
]
].head(20)
)

# ------------------------------------------------------------
# 19. SAVE PREDICTIONS
# ------------------------------------------------------------

test_results_df.to_csv(
"bert_sentiment_predictions.csv",
index=False
)

print(
"\nPredictions saved as "
"bert_sentiment_predictions.csv"
)

# ------------------------------------------------------------
# 20. URGENCY DETECTION
# ------------------------------------------------------------

# In a social-impact/help-line setting,
# distress and negative cases can be prioritized.

def calculate_urgency(sentiment):

if sentiment == "distress":
return "HIGH"

elif sentiment == "negative":
return "MEDIUM"

elif sentiment == "neutral":
return "LOW"

else:
return "LOW"


test_results_df["urgency"] = (
test_results_df["predicted_label"]
.apply(calculate_urgency)
)

print("\nUrgency distribution:")

print(
test_results_df["urgency"].value_counts()
)


In [ ]:

# ------------------------------------------------------------
# 21. PRIORITIZE CASES
# ------------------------------------------------------------

urgency_order = {
"HIGH": 0,
"MEDIUM": 1,
"LOW": 2
}

test_results_df["urgency_rank"] = (
test_results_df["urgency"]
.map(urgency_order)
)

priority_cases = test_results_df.sort_values(
by="urgency_rank"
)

print("\nTop priority cases:")

print(
priority_cases[
[
TEXT_COLUMN,
"predicted_label",
"urgency"
]
].head(20)
)

# ------------------------------------------------------------
# 22. SAVE PRIORITIZED CASES
# ------------------------------------------------------------

priority_cases.to_csv(
"prioritized_cases.csv",
index=False
)

print(
"\nPrioritized cases saved as "
"prioritized_cases.csv"
)

# ------------------------------------------------------------
# 23. AUTOMATIC SINGLE-TEXT PREDICTION
# ------------------------------------------------------------

def predict_sentiment(text):

inputs = tokenizer(
text,
return_tensors="pt",
truncation=True,
padding=True,
max_length=128
)

# Move inputs to same device as model
device = model.device

inputs = {
key: value.to(device)
for key, value in inputs.items()
}

with torch.no_grad():

outputs = model(**inputs)

probabilities = torch.softmax(
outputs.logits,
dim=1
)

predicted_class = torch.argmax(
probabilities,
dim=1
).item()

confidence = probabilities[
0,
predicted_class
].item()

sentiment = id2label[
predicted_class
]

urgency = calculate_urgency(
sentiment
)

return {
"text": text,
"sentiment": sentiment,
"confidence": confidence,
"urgency": urgency
}


# ------------------------------------------------------------
# 24. TEST THE MODEL ON NEW FEEDBACK
# ------------------------------------------------------------

new_feedback = [
"Thank you for helping my family. We really appreciate the support.",
"I am extremely worried and need help immediately.",
"The situation has not changed and I am waiting for assistance.",
"The service was disappointing and nobody responded to my request."
]

print("\nNew feedback predictions:")

for text in new_feedback:

result = predict_sentiment(text)

print("\nText:", result["text"])

print(
"Sentiment:",
result["sentiment"]
)

print(
"Confidence:",
f"{result['confidence'] * 100:.2f}%"
)

print(
"Urgency:",
result["urgency"]
)

# ------------------------------------------------------------
# 25. SAVE THE TRAINED MODEL
# ------------------------------------------------------------

print("\nSaving trained BERT model...")

trainer.save_model(
"./final_bert_sentiment_model"
)

tokenizer.save_pretrained(
"./final_bert_sentiment_model"
)

print(
"Model saved to "
"./final_bert_sentiment_model"
)

# ------------------------------------------------------------
# 26. RESPONSE-TIME ANALYSIS
# ------------------------------------------------------------

# If your dataset contains response-time information,
# you can compare manual review against automated triage.

# Expected columns:
#
# manual_response_time
# automated_response_time
#
# Example:
#
# response_df = pd.read_csv("response_times.csv")

def calculate_response_improvement(
manual_time,
automated_time
):

improvement = (
(manual_time - automated_time)
/ manual_time
) * 100

return improvement


# Example calculation:
#
# manual_average = response_df[
# "manual_response_time"
# ].mean()
#
# automated_average = response_df[
# "automated_response_time"
# ].mean()
#
# improvement = calculate_response_improvement(
# manual_average,
# automated_average
# )
#
# print(
# f"Response time improvement: "
# f"{improvement:.2f}%"
# )

# ------------------------------------------------------------
# END OF PROJECT
# ------------------------------------------------------------

print("\n======================================")
print("PROJECT COMPLETED")
print("======================================")

print(
f"BERT Test Accuracy: "
f"{test_accuracy * 100:.2f}%"
)

print(
"Classes:",
", ".join(LABEL_NAMES)
)

print(
"Model:",
MODEL_NAME
)

print("======================================")